In [27]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
from langchain_teddynote import logging

logging.langsmith("Summary")

LangSmith 추적을 시작합니다.
[프로젝트명]
Summary


In [29]:
#Stuff: 문서 목록을 가져와서 모두 프롬프트에 삽입하고 그 프롬포트를 llm에 전달
#문서가 작고 대부분의 호출에 몇개반 전달되는 애플리케이션에 적합

from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/news.txt", encoding="utf-8")
docs = loader.load()
print(f"총 글자수: {len(docs[0].page_content)}")
print("\n=========앞부분 미리보기=========\n")
print(docs[0].page_content[:500])

총 글자수: 1708

=========앞부분 미리보기=========

제목: 
AI2, 상업 활용까지 자유로운 '진짜' 오픈 소스 LLM '올모' 출시

내용:
앨런AI연구소(AI2)가 완전한 오픈 소스 대형언어모델(LLM) '올모(OLMo)’를 출시했다. 데이터 수집, 학습, 배포의 전 과정을 투명하게 공개한 데다 상업적 사용까지 허용한 진정한 의미의 오픈 소스 LLM이라는 평가다.
벤처비트는 1일(현지시간) 비영리 민간 AI 연구기관인 AI2가 ‘최초의 진정한 오픈 소스 LLM 및 프레임워크’라고 소개한 ‘올모’를 출시했다고 보도했다. 
이에 따르면 올모는 모델 코드와 모델 가중치뿐만 아니라 훈련 코드, 훈련 데이터, 관련 툴킷 및 평가 툴킷도 제공한다. 이를 통해 모델이 어떻게 구축되었는지 심층적으로 분석, LLM의 작동 방식과 응답을 생성하는 원리를 더 잘 이해할 수 있다. 
올모 프레임워크는 70억 매개변수의 ‘올모 7B’ 등 4가지 변형 모델과 10억 매개변수의 ‘올모 1B’ 모델을 제공한다. 모델들은 훈련 데이터를 생성하는 코드를 포함해 


In [30]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "teddynote/summary-stuff-documents-korean",
    dangerously_pull_public_prompt=True
)

prompt.pretty_print()

Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:"



In [31]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. Translate the summary into KOREAN if it is written in ENGLISH.
5. DO NOT translate any technical terms.
6. DO NOT include any unnecessary information.
7. Each bullet point must contain only one sentence.
8. DO NOT use Markdown bold syntax such as **text**.
9. DO NOT use asterisks (*) in the output.
10. DO NOT add additional explanations to the bullet points.
11. Regarding nouns, include english phrases right after Korean. ex) 올모(OLMo)

CONTEXT:
{context}

SUMMARY:"
"""
)

In [32]:
from langchain_openai import ChatOpenAI
from langchain_teddynote.callbacks import StreamingCallback


llm = ChatOpenAI(
    model="gpt-4o-mini",
    streaming=True,
    temperature=0,
    callbacks=[StreamingCallback()],
)

chain = prompt | llm

answer = chain.invoke({
    "context": docs
})

- 🚀 AI2가 완전한 오픈 소스 LLM인 올모(OLMo)를 출시했다.
- 📊 데이터 수집, 학습, 배포 과정을 투명하게 공개하고 상업적 사용을 허용한다.
- 🛠️ 올모는 모델 코드, 가중치, 훈련 코드, 데이터 및 툴킷을 제공한다.
- 📈 70억 매개변수의 올모 7B와 10억 매개변수의 올모 1B 모델이 포함되어 있다.
- 📚 AI2의 돌마(Dolma) 데이터 세트를 기반으로 3조개의 토큰으로 훈련되었다.
- 🔍 전체 모델 가중치와 훈련 로그, 평가 제품군이 포함되어 있다.
- 💼 아파치 2.0 라이선스에 따라 상업적 활용에 제한이 없다.
- 🧪 올모 프로젝트 책임자는 투명성이 부족한 현재의 언어 모델에 대한 문제를 지적했다.
- 🌟 올모는 상업용 제품과 동등한 성능을 보여주고 있다.
- 🌍 비영어권 언어에 대한 낮은 품질과 약한 코드 생성 기능이 있다.
- 🔄 AI2는 올모를 계속해서 향상할 계획이다.
- 🏗️ 올모의 출시는 AI 모델에 대한 깊은 이해를 위한 기반을 구축하는 것이다.
- 🤝 오픈 소스 기반 모델은 AI 혁신과 개발에 중요하다는 의견이 있다.
- 📥 올모의 모든 리소스는 깃허브 및 허깅페이스에서 무료로 제공된다.

In [33]:
#Map-Reduce
#문서를 작은 chunk 단계로 나누는 map 단계와 chunk의 요약을 결합하는 reduce 단계로 구성

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/SPRI_AI_Brief_2023년12월호_F.pdf")
docs = loader.load()
docs = docs[3:8]
print(f"총 페이지수: {len(docs)}")

총 페이지수: 5


In [39]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
)



map_prompt = ChatPromptTemplate.from_template("""
다음 문서의 핵심 내용을 요약해주세요.

문서:
{docs}

요약:
""")



# Map Chain 생성
map_chain = (
    RunnableLambda(lambda doc: {"docs": doc.page_content})
    | map_prompt
    | llm
    | StrOutputParser()
)



doc_summaries = map_chain.batch(docs)

print(len(doc_summaries))
print(doc_summaries[0])

5
미국 바이든 대통령은 2023년 10월 30일 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령'에 서명하여 AI의 안전성과 신뢰성을 보장하기 위한 광범위한 조치를 발표했다. 이 행정명령은 AI의 안전과 보안 기준 마련, 개인정보 보호, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 주요 내용으로 포함하고 있다.

주요 내용은 다음과 같다:
1. **AI 안전과 보안 기준**: 기업은 AI 시스템의 안전 테스트 결과와 주요 정보를 정부와 공유해야 하며, AI 생성 콘텐츠에 대한 표준과 모범 사례를 확립할 예정이다.
2. **형평성과 시민권 향상**: AI의 차별적 사용을 방지하기 위한 법률 및 지침을 마련하고, 형사사법 및 주택 임대 분야에서의 AI 사용 모범사례를 개발한다.
3. **소비자 보호와 근로자 지원**: 의료 분야에서 책임 있는 AI 사용을 촉진하고, AI로 인한 근로자 피해를 완화하기 위한 원칙과 모범사례를 마련한다.
4. **혁신과 경쟁 촉진**: 국가 AI 연구자원(NAIRR)을 통해 AI 연구를 지원하고, 외국인 전문가들이 미국에서 공부하고 일할 수 있도록 비자 기준을 현대화한다.

이 행정명령은 AI 기술의 안전하고 책임 있는 발전을 도모하고, 미국 내 AI 산업의 경쟁력을 강화하는 데 기여할 것으로 기대된다.


In [40]:
#reduce prompt 생성

reduce_prompt = ChatPromptTemplate.from_template("""
다음은 여러 문서를 각각 요약한 내용입니다.

이 요약들을 하나로 통합하여 최종 요약을 작성해 주세요.

각 문서의 요약:
{doc_summaries}

최종 요약:
""")



In [41]:
reduce_llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
)

In [42]:
reduce_chain = (
    reduce_prompt
    | reduce_llm
    |StrOutputParser()
)

In [43]:
answer = reduce_chain.invoke({
    "doc_summaries": "/n".join(doc_summaries)
})

print(answer)

2023년 10월 30일, 미국 바이든 대통령은 '안전하고 신뢰할 수 있는 AI 개발과 사용에 관한 행정명령'에 서명하여 AI의 안전성과 신뢰성을 보장하기 위한 조치를 발표했다. 이 행정명령은 AI 안전과 보안 기준, 형평성과 시민권 향상, 소비자 보호, 노동자 지원, 혁신과 경쟁 촉진, 국제 협력을 주요 내용으로 포함하고 있다. 기업은 AI 시스템의 안전 테스트 결과를 정부와 공유해야 하며, AI의 차별적 사용을 방지하기 위한 법률과 지침이 마련될 예정이다.

같은 날, G7은 '히로시마 AI 프로세스'를 통해 AI 시스템 개발을 위한 국제 행동강령에 합의하고, AI 위험 식별 및 완화를 위한 자발적 채택을 권장했다. 이 행동강령은 AI 수명주기 전반에 걸친 위험 평가와 투명성 보장 등을 포함하며, 국제 기술 표준 개발을 촉진할 계획이다.

2023년 11월 1~2일 영국 블레츨리 파크에서 열린 AI 안전성 정상회의에서는 28개국이 AI 안전 보장을 위한 '블레츨리 선언'을 발표하고, AI 위험 관리를 위한 협력의 중요성을 강조했다. 영국은 AI 안전 연구소 출범과 정부 주도의 안전 테스트 계획을 발표했으며, 참가국들은 AI 위험에 대한 과학적 평가를 위한 보고서 작성에 합의했다.

한편, 미국 캘리포니아 북부지방법원은 예술가들이 AI 기업을 상대로 제기한 저작권 침해 소송을 기각했으며, 연방거래위원회(FTC)는 생성 AI의 소비자와 창작자에 대한 피해 우려를 표명하고, 시장 지배력 강화에 대한 경고를 했다. FTC는 소비자 보호와 공정한 경쟁 유지를 위해 모든 권한을 활용할 것이라고 강조했다. 이러한 일련의 조치는 AI 기술의 안전하고 책임 있는 발전을 도모하고, 글로벌 AI 거버넌스와 위험 관리 정책을 강화하는 데 기여할 것으로 기대된다.


In [ ]:
#map-refine: 생성된 요약을 순차적으로 처리하며 최종 요약을 점진적으로 개선. 이전 요약과 새로운 chunk의 정보를 결합하여 요약 갱신 


